
# URLFusionNet — Fixed-Average Multi-Seed Control Experiment

This notebook performs the **one additional experiment required before locking the Results section**:

> Compare the proposed **sample-adaptive character–token fusion** against **fixed-average fusion** under the **same seeds and training protocol**.

The previous final experiment already evaluated the proposed model using seeds **42, 123, and 2025**. Therefore, by default this notebook **does not retrain the proposed model**. It trains only `fixed_avg` with the same:

- cleaned Kaggle Malicious URLs Dataset;
- fixed 80:10:10 stratified split;
- training-only vectorizers;
- class weights;
- model encoders and residual refinement;
- AdamW optimizer;
- learning rate `8e-4`;
- weight decay `1e-4`;
- gradient clipping `clipnorm=1.0`;
- maximum `6` epochs;
- validation Macro-F1 model selection;
- EarlyStopping patience `2`;
- ReduceLROnPlateau patience `1`;
- seeds **42, 123, 2025**.

The notebook produces a **paired per-seed comparison**, mean ± standard deviation, stability analysis, and a results-ready table.

> Set `RERUN_PROPOSED = True` only if you want to repeat both models from scratch. The default `False` saves substantial runtime and uses the actual proposed-model results already produced by the final experiment.


In [ ]:
# Dependencies are installed in the isolated WSL2 CUDA environment.
import sys
print("Kernel Python:", sys.executable)
assert "ep_hi_cuda_linux/venv" in sys.executable, "Select EPHI TensorFlow CUDA (Linux Disk) kernel."


## 1. Configuration — identical to the final experiment

In [ ]:

import os, gc, glob, time, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    matthews_corrcoef, classification_report
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Bound TensorFlow CPU overhead so existing Jupyter experiments can continue.
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(2)

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ---------------- FINAL CONTROL PROTOCOL ----------------
SPLIT_SEED = 42
SEEDS = [42, 123, 2025]
MAIN_EPOCHS = 6

# False = train only FixedAvg and compare with previously obtained proposed results.
# True  = rerun both Proposed and FixedAvg for all three seeds.
RERUN_PROPOSED = True

# If a seed-level CSV already exists, it is reused unless FORCE_RETRAIN=True.
FORCE_RETRAIN = False

gpus = tf.config.list_physical_devices("GPU")
assert gpus, "CUDA NOT AVAILABLE: Select WSL2 EPHI CUDA Python kernel."
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

BATCH_SIZE = 256  # same as original CPU protocol; safe for 8 GB RTX
LEARNING_RATE = 8e-4

MAX_CHAR_LEN = 200
MAX_TOKEN_LEN = 40
MAX_CHAR_VOCAB = 256
MAX_TOKEN_VOCAB = 30000
CHAR_EMB_DIM = 32
TOKEN_EMB_DIM = 48
CONV_FILTERS = 48
LATENT_DIM = 96
ATTN_HEADS = 2
DROPOUT = 0.20
NUM_CLASSES = 4

CLASS_NAMES = ["benign", "defacement", "phishing", "malware"]
LABEL_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}
ID_TO_LABEL = {i: n for n, i in LABEL_TO_ID.items()}

RESULT_DIR=Path(os.environ.get('URLFUSIONNET_RESULT_DIR', str(Path.cwd() / 'outputs' / 'fixedavg'))).expanduser().resolve()
RESULT_DIR.mkdir(parents=True,exist_ok=True)

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_all_seeds(42)

print("TensorFlow:", tf.__version__)
print("GPU(s):", gpus)
print("Batch size:", BATCH_SIZE)
print("Seeds:", SEEDS)
print("Epochs:", MAIN_EPOCHS)
print("RERUN_PROPOSED:", RERUN_PROPOSED)


## 2. Load Kaggle dataset, audit duplicates/conflicts, and reproduce the same split

In [ ]:

KAGGLE_DATASET="sid321axn/malicious-urls-dataset"
EXPECTED_COUNTS={
    "benign":428103,
    "defacement":96457,
    "phishing":94111,
    "malware":32520,
}

candidates=[
    "/kaggle/input/malicious-urls-dataset/malicious_phish.csv",
    "./malicious_phish.csv",
    "data/raw/malicious_phish.csv",
    "../data/raw/malicious_phish.csv",
    "malicious_phish.csv",
]
manual_dataset=os.environ.get('URLFUSIONNET_DATASET_CSV')
if manual_dataset and not Path(manual_dataset).is_file():
    raise FileNotFoundError('URLFUSIONNET_DATASET_CSV does not point to a file')
csv_path=manual_dataset or next((p for p in candidates if os.path.exists(p)),None)

if csv_path is None:
    import kagglehub
    root=kagglehub.dataset_download(KAGGLE_DATASET)
    matches=glob.glob(os.path.join(root,"**","malicious_phish.csv"),recursive=True)
    if not matches:
        matches=glob.glob(os.path.join(root,"**","*.csv"),recursive=True)
    if not matches: raise FileNotFoundError("Dataset CSV not found.")
    csv_path=matches[0]

raw=pd.read_csv(csv_path)
cols={c.lower().strip():c for c in raw.columns}
url_col=next((cols[k] for k in ["url","urls","link"] if k in cols),None)
label_col=next((cols[k] for k in ["type","label","class","category"] if k in cols),None)

df=raw[[url_col,label_col]].copy()
df.columns=["url","label"]
df["url"]=df["url"].astype("string")
df["label"]=df["label"].astype("string").str.strip().str.lower()

observed={k:int(df["label"].value_counts().get(k,0)) for k in EXPECTED_COUNTS}
assert observed==EXPECTED_COUNTS, observed

df=df.dropna(subset=["url","label"]).copy()
df["url"]=df["url"].str.strip()
df=df[df["url"].str.len()>0]
df=df[df["label"].isin(CLASS_NAMES)].copy()

card=df.groupby("url",sort=False)["label"].nunique()
conflict_urls=card[card>1].index
df=df[~df["url"].isin(conflict_urls)].copy()
df=df.drop_duplicates(subset=["url","label"],keep="first").copy()
df["label_id"]=df["label"].map(LABEL_TO_ID).astype("int32")

train_df,temp_df=train_test_split(
    df,test_size=0.20,random_state=SPLIT_SEED,stratify=df["label_id"]
)
val_df,test_df=train_test_split(
    temp_df,test_size=0.50,random_state=SPLIT_SEED,stratify=temp_df["label_id"]
)
for x in [train_df,val_df,test_df]: x.reset_index(drop=True,inplace=True)

assert len(set(train_df.url)&set(val_df.url))==0
assert len(set(train_df.url)&set(test_df.url))==0
assert len(set(val_df.url)&set(test_df.url))==0

counts=train_df["label_id"].value_counts().sort_index()
CLASS_WEIGHT={
    k:float(len(train_df)/(NUM_CLASSES*counts.loc[k]))
    for k in range(NUM_CLASSES)
}

print("Clean:",len(df))
print("Train/Val/Test:",len(train_df),len(val_df),len(test_df))
print("Class weights:",CLASS_WEIGHT)


## 3. Training-only character/token vectorizers and tf.data pipelines

In [ ]:

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
def url_token_standardize(x):
    x=tf.strings.lower(x)
    x=tf.strings.regex_replace(x,r"[^a-z0-9]+"," ")
    return tf.strings.strip(x)

char_vectorizer=layers.TextVectorization(
    max_tokens=MAX_CHAR_VOCAB,standardize=None,split="character",
    output_mode="int",output_sequence_length=MAX_CHAR_LEN
)
token_vectorizer=layers.TextVectorization(
    max_tokens=MAX_TOKEN_VOCAB,standardize=url_token_standardize,
    split="whitespace",output_mode="int",
    output_sequence_length=MAX_TOKEN_LEN
)

adapt_ds=tf.data.Dataset.from_tensor_slices(
    train_df["url"].astype(str).values
).batch(4096)
char_vectorizer.adapt(adapt_ds)
token_vectorizer.adapt(adapt_ds)

AUTOTUNE=tf.data.AUTOTUNE

def make_dataset(frame,training=False,seed=42):
    ds=tf.data.Dataset.from_tensor_slices((
        frame["url"].astype(str).values,
        frame["label_id"].astype("int32").values
    ))
    if training:
        ds=ds.shuffle(min(len(frame),200000),seed=seed,reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def make_predict_dataset(urls):
    return tf.data.Dataset.from_tensor_slices(
        np.asarray(urls,dtype=str)
    ).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds=make_dataset(val_df,False)


## 4. Stable custom layers and Macro-F1

In [ ]:

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class PositionalEmbedding(layers.Layer):
    def __init__(self,max_length,dim,**kwargs):
        super().__init__(**kwargs); self.max_length=max_length; self.dim=dim
    def build(self,input_shape):
        self.positional=self.add_weight(
            name="positional_embedding",
            shape=(1,self.max_length,self.dim),
            initializer="random_normal",trainable=True
        )
    def call(self,x):
        return x+self.positional[:,:tf.shape(x)[1],:]

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class AbsDifference(layers.Layer):
    def call(self,inputs):
        a,b=inputs; return tf.abs(a-b)

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class AdaptiveWeightedFusion(layers.Layer):
    def call(self,inputs):
        zc,zt,a=inputs
        return a[:,0:1]*zc+a[:,1:2]*zt

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class SafeMaskedMean(layers.Layer):
    def call(self,inputs):
        x,m=inputs
        m=tf.cast(m,x.dtype)[...,None]
        return tf.reduce_sum(x*m,axis=1)/tf.maximum(
            tf.reduce_sum(m,axis=1),tf.cast(1.0,x.dtype)
        )

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class SparseMacroF1(tf.keras.metrics.Metric):
    def __init__(self,num_classes=4,name="macro_f1",**kwargs):
        super().__init__(name=name,**kwargs)
        self.num_classes=num_classes
        self.cm=self.add_weight(
            name="cm",shape=(num_classes,num_classes),
            initializer="zeros",dtype=tf.float32
        )
    def update_state(self,y_true,y_pred,sample_weight=None):
        yt=tf.cast(tf.reshape(y_true,[-1]),tf.int32)
        yp=tf.argmax(y_pred,axis=-1,output_type=tf.int32)
        self.cm.assign_add(tf.math.confusion_matrix(
            yt,yp,num_classes=self.num_classes,dtype=tf.float32
        ))
    def result(self):
        tp=tf.linalg.diag_part(self.cm)
        fp=tf.reduce_sum(self.cm,axis=0)-tp
        fn=tf.reduce_sum(self.cm,axis=1)-tp
        p=tf.math.divide_no_nan(tp,tp+fp)
        r=tf.math.divide_no_nan(tp,tp+fn)
        f=tf.math.divide_no_nan(2*p*r,p+r)
        return tf.reduce_mean(f)
    def reset_state(self):
        self.cm.assign(tf.zeros_like(self.cm))



## 5. Exact architecture used in the final experiment

For this control experiment, **all representation branches and residual refinement remain identical**. The only difference is the fusion rule:

Proposed adaptive fusion:

\[
\mathbf{f}_i =
\alpha_i^c \mathbf{z}_i^c +
\alpha_i^t \mathbf{z}_i^t
\]

Fixed-average control:

\[
\mathbf{f}_i =
\frac{1}{2}\mathbf{z}_i^c +
\frac{1}{2}\mathbf{z}_i^t.
\]

Thus, the experiment isolates whether the learned sample-specific gate contributes beyond simply combining both granularities.


In [ ]:

def build_model(variant="proposed"):
    url=keras.Input(shape=(),dtype=tf.string,name="url")

    # character branch
    cids=char_vectorizer(url)
    xc=layers.Embedding(
        len(char_vectorizer.get_vocabulary()),CHAR_EMB_DIM,name="char_embedding"
    )(cids)
    pools=[]
    for k in (3,5,7):
        h=layers.Conv1D(CONV_FILTERS,k,padding="same",use_bias=False,
                        name=f"char_conv_k{k}")(xc)
        h=layers.BatchNormalization(name=f"char_bn_k{k}")(h)
        h=layers.Activation(tf.nn.gelu,name=f"char_gelu_k{k}")(h)
        pools.append(layers.GlobalMaxPooling1D(name=f"char_gmp_k{k}")(h))
    hc=layers.Concatenate(name="char_multiscale_concat")(pools)
    hc=layers.Dense(LATENT_DIM,name="char_scale_projection")(hc)

    # token branch
    tids=token_vectorizer(url)
    token_mask=layers.Lambda(
        lambda ids:tf.not_equal(ids,0),name="token_valid_mask"
    )(tids)
    xt=layers.Embedding(
        len(token_vectorizer.get_vocabulary()),TOKEN_EMB_DIM,
        mask_zero=False,name="token_embedding"
    )(tids)
    xt=PositionalEmbedding(MAX_TOKEN_LEN,TOKEN_EMB_DIM,name="token_position")(xt)
    key_mask=layers.Lambda(
        lambda m:tf.expand_dims(m,axis=1),name="token_key_mask"
    )(token_mask)
    attn=layers.MultiHeadAttention(
        num_heads=ATTN_HEADS,key_dim=TOKEN_EMB_DIM//ATTN_HEADS,
        dropout=0.10,name="token_self_attention"
    )(xt,xt,attention_mask=key_mask)
    xt=layers.LayerNormalization(name="token_attn_ln")(layers.Add()([xt,attn]))
    ff=layers.Dense(TOKEN_EMB_DIM*2,activation=tf.nn.gelu,name="token_ff1")(xt)
    ff=layers.Dropout(DROPOUT)(ff)
    ff=layers.Dense(TOKEN_EMB_DIM,name="token_ff2")(ff)
    xt=layers.LayerNormalization(name="token_ff_ln")(layers.Add()([xt,ff]))
    ht=SafeMaskedMean(name="token_safe_pool")([xt,token_mask])

    # common space
    zc=layers.LayerNormalization(name="z_char")(
        layers.Dense(LATENT_DIM,name="char_common_projection")(hc)
    )
    zt=layers.LayerNormalization(name="z_token")(
        layers.Dense(LATENT_DIM,name="token_common_projection")(ht)
    )

    if variant=="char_only":
        fused=zc
    elif variant=="token_only":
        fused=zt
    elif variant=="fixed_avg":
        fused=layers.Average(name="fixed_average_fusion")([zc,zt])
    else:
        if variant=="explicit_interaction":
            prod=layers.Multiply(name="cross_product")([zc,zt])
            diff=AbsDifference(name="cross_absdiff")([zc,zt])
            gate_input=layers.Concatenate(name="interaction_representation")(
                [zc,zt,prod,diff]
            )
        else:
            gate_input=layers.Concatenate(name="adaptive_gate_context")([zc,zt])

        gh=layers.Dense(
            LATENT_DIM//2,activation=tf.nn.gelu,name="gate_hidden"
        )(gate_input)
        alpha=layers.Dense(
            2,activation="softmax",dtype="float32",name="fusion_weights"
        )(gh)
        fused=AdaptiveWeightedFusion(name="adaptive_weighted_fusion")(
            [zc,zt,alpha]
        )

    use_residual = variant not in (
        "char_only","token_only","proposed_no_residual"
    )
    if use_residual:
        r=layers.Dense(LATENT_DIM*2,activation=tf.nn.gelu,name="residual_fc1")(fused)
        r=layers.Dropout(DROPOUT)(r)
        r=layers.Dense(LATENT_DIM,name="residual_fc2")(r)
        fused=layers.LayerNormalization(name="residual_ln")(layers.Add()([fused,r]))

    out=layers.Dense(
        NUM_CLASSES,activation="softmax",dtype="float32",name="classifier"
    )(fused)
    return keras.Model(url,out,name=f"URLFusionNet_{variant}")

def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=LEARNING_RATE,weight_decay=1e-4,clipnorm=1.0
        ),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            SparseMacroF1(NUM_CLASSES,name="macro_f1")
        ]
    )
    return model


## 6. Training/evaluation helpers — same optimizer and selection protocol

In [ ]:

def callbacks(tag):
    checkpoint_path = RESULT_DIR / f"best_{tag}.weights.h5"
    return [
        keras.callbacks.TerminateOnNaN(),
        keras.callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor="val_macro_f1",
            mode="max",
            save_best_only=True,
            save_weights_only=True,
            verbose=0,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_macro_f1",
            mode="max",
            patience=2,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_macro_f1",
            mode="max",
            factor=0.5,
            patience=1,
            min_lr=1e-5,
            verbose=1,
        ),
        keras.callbacks.CSVLogger(
            RESULT_DIR / f"history_{tag}.csv"
        ),
    ]


def train_variant(variant, seed, epochs=MAIN_EPOCHS):
    tf.keras.backend.clear_session()
    set_all_seeds(seed)

    model = compile_model(build_model(variant))
    history = model.fit(
        make_dataset(train_df, True, seed),
        validation_data=val_ds,
        epochs=epochs,
        class_weight=CLASS_WEIGHT,
        callbacks=callbacks(f"{variant}_seed{seed}"),
        verbose=1,
    )
    return model, history


def metric_row(y_true, y_pred, name, seed):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "Model": name,
        "Seed": seed,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_Precision": p,
        "Macro_Recall": r,
        "Macro_F1": f,
        "Weighted_F1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }


def evaluate(model, frame, name, seed, save_report=True):
    y_true = frame["label_id"].to_numpy()
    probs = model.predict(
        make_predict_dataset(frame["url"].astype(str).values),
        verbose=0,
    )
    assert np.isfinite(probs).all(), f"{name}, seed={seed}: non-finite probabilities."
    y_pred = probs.argmax(axis=1)

    row = metric_row(y_true, y_pred, name, seed)

    if save_report:
        report = classification_report(
            y_true,
            y_pred,
            target_names=CLASS_NAMES,
            output_dict=True,
            digits=6,
            zero_division=0,
        )
        pd.DataFrame(report).T.to_csv(
            RESULT_DIR / f"classification_report_{name}_seed{seed}.csv"
        )

    return row, probs, y_pred


def history_selection_summary(history, model_name, seed):
    h = pd.DataFrame(history.history)
    best_idx = int(h["val_macro_f1"].idxmax())
    return {
        "Model": model_name,
        "Seed": seed,
        "Epochs_Executed": len(h),
        "Best_Epoch": best_idx + 1,
        "Best_Val_Macro_F1": float(h.loc[best_idx, "val_macro_f1"]),
        "Best_Val_Accuracy": float(h.loc[best_idx, "val_accuracy"]),
        "Best_Val_Loss": float(h.loc[best_idx, "val_loss"]),
    }



## 7. Locked proposed-model reference from the completed FINAL run

These are the **actual outputs** from the immediately preceding final experiment. They are used only as the paired reference when `RERUN_PROPOSED=False`.


In [ ]:

PROPOSED_REFERENCE = pd.DataFrame([
    {
        "Model": "URLFusionNet-Adaptive",
        "Seed": 42,
        "Accuracy": 0.9837627901172947,
        "Macro_Precision": 0.9767413578928437,
        "Macro_Recall": 0.9754182754637868,
        "Macro_F1": 0.9759701432434308,
        "Weighted_F1": 0.9838509286106419,
        "MCC": 0.9682489019864169,
    },
    {
        "Model": "URLFusionNet-Adaptive",
        "Seed": 123,
        "Accuracy": 0.9827957324681806,
        "Macro_Precision": 0.9715263202771973,
        "Macro_Recall": 0.9783478836453073,
        "Macro_F1": 0.9747636821683450,
        "Weighted_F1": 0.9829660442973435,
        "MCC": 0.9666006169318198,
    },
    {
        "Model": "URLFusionNet-Adaptive",
        "Seed": 2025,
        "Accuracy": 0.9336941602196157,
        "Macro_Precision": 0.9164721236247478,
        "Macro_Recall": 0.9523262413030280,
        "Macro_F1": 0.9297621878968372,
        "Weighted_F1": 0.9372261635785443,
        "MCC": 0.8806313458552558,
    },
])

display(PROPOSED_REFERENCE)
PROPOSED_REFERENCE.to_csv(
    RESULT_DIR / "proposed_reference_previous_final_run.csv",
    index=False,
)

print("\nProposed previous-run mean ± std:")
display(
    PROPOSED_REFERENCE
    .drop(columns=["Model", "Seed"])
    .agg(["mean", "std"])
    .T
)


## 8. Main requested experiment — Fixed-average fusion, seeds 42/123/2025

In [ ]:

fixed_rows = []
selection_rows = []

for seed in SEEDS:
    seed_csv = RESULT_DIR / f"fixed_avg_seed{seed}_metrics.csv"

    if seed_csv.exists() and not FORCE_RETRAIN:
        print(f"\nSeed {seed}: reusing completed result -> {seed_csv}")
        cached = pd.read_csv(seed_csv).iloc[0].to_dict()
        fixed_rows.append(cached)
        continue

    print("\n" + "=" * 84)
    print(f"TRAINING FIXED-AVERAGE FUSION | seed={seed} | epochs<= {MAIN_EPOCHS}")
    print("=" * 84)

    model, history = train_variant(
        variant="fixed_avg",
        seed=seed,
        epochs=MAIN_EPOCHS,
    )

    row, probs, pred = evaluate(
        model,
        test_df,
        name="Fixed-average fusion",
        seed=seed,
        save_report=True,
    )

    fixed_rows.append(row)
    selection_rows.append(
        history_selection_summary(
            history,
            model_name="Fixed-average fusion",
            seed=seed,
        )
    )

    pd.DataFrame([row]).to_csv(seed_csv, index=False)
    np.save(
        RESULT_DIR / f"fixed_avg_seed{seed}_test_predictions.npy",
        pred,
    )
    np.save(
        RESULT_DIR / f"fixed_avg_seed{seed}_test_probabilities.npy",
        probs,
    )
    model.save_weights(
        RESULT_DIR / f"FixedAvg_seed{seed}.weights.h5"
    )

    del model, history, probs, pred
    gc.collect()

fixed_df = pd.DataFrame(fixed_rows)
fixed_df["Seed"] = fixed_df["Seed"].astype(int)
fixed_df = fixed_df.sort_values("Seed").reset_index(drop=True)

display(fixed_df)
fixed_df.to_csv(
    RESULT_DIR / "fixedavg_multiseed_raw.csv",
    index=False,
)

if selection_rows:
    selection_df = pd.DataFrame(selection_rows)
    display(selection_df)
    selection_df.to_csv(
        RESULT_DIR / "fixedavg_training_selection_summary.csv",
        index=False,
    )


## 9. Fixed-average multi-seed stability

In [ ]:

metric_cols = [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Weighted_F1",
    "MCC",
]

fixed_summary = fixed_df[metric_cols].agg(["mean", "std"]).T
fixed_summary["CV_percent"] = (
    100.0 * fixed_summary["std"] / fixed_summary["mean"].abs()
)

display(fixed_summary)
fixed_summary.to_csv(
    RESULT_DIR / "fixedavg_multiseed_mean_std.csv"
)

print(
    f"FixedAvg Macro-F1 = "
    f"{fixed_summary.loc['Macro_F1','mean']:.6f} ± "
    f"{fixed_summary.loc['Macro_F1','std']:.6f}"
)



## 10. Optional exact rerun of the proposed model

The default is `RERUN_PROPOSED=False` because the proposed model has already been run with the identical final protocol. Set it to `True` only if you want a fresh paired rerun.


In [ ]:

if RERUN_PROPOSED:
    proposed_rows = []

    for seed in SEEDS:
        seed_csv = RESULT_DIR / f"proposed_rerun_seed{seed}_metrics.csv"

        if seed_csv.exists() and not FORCE_RETRAIN:
            print(f"Seed {seed}: reusing proposed rerun result.")
            proposed_rows.append(pd.read_csv(seed_csv).iloc[0].to_dict())
            continue

        print("\n" + "=" * 84)
        print(f"RERUN PROPOSED ADAPTIVE FUSION | seed={seed}")
        print("=" * 84)

        model, history = train_variant(
            variant="proposed",
            seed=seed,
            epochs=MAIN_EPOCHS,
        )

        row, probs, pred = evaluate(
            model,
            test_df,
            name="URLFusionNet-Adaptive",
            seed=seed,
            save_report=True,
        )
        proposed_rows.append(row)
        pd.DataFrame([row]).to_csv(seed_csv, index=False)

        np.save(
            RESULT_DIR / f"proposed_rerun_seed{seed}_test_predictions.npy",
            pred,
        )
        model.save_weights(
            RESULT_DIR / f"Proposed_rerun_seed{seed}.weights.h5"
        )

        del model, history, probs, pred
        gc.collect()

    proposed_df = pd.DataFrame(proposed_rows)
    proposed_df["Seed"] = proposed_df["Seed"].astype(int)
    proposed_df = proposed_df.sort_values("Seed").reset_index(drop=True)
else:
    proposed_df = PROPOSED_REFERENCE.copy()

display(proposed_df)


## 11. Paired seed-by-seed Adaptive vs FixedAvg comparison

In [ ]:

proposed_pair = proposed_df[
    ["Seed"] + metric_cols
].copy()

fixed_pair = fixed_df[
    ["Seed"] + metric_cols
].copy()

paired = proposed_pair.merge(
    fixed_pair,
    on="Seed",
    suffixes=("_Adaptive", "_FixedAvg"),
    validate="one_to_one",
)

for metric in metric_cols:
    paired[f"Delta_{metric}"] = (
        paired[f"{metric}_Adaptive"]
        - paired[f"{metric}_FixedAvg"]
    )

display(paired)
paired.to_csv(
    RESULT_DIR / "adaptive_vs_fixed_paired_by_seed.csv",
    index=False,
)

print("\nMacro-F1 paired differences (Adaptive - FixedAvg):")
for _, r in paired.iterrows():
    print(
        f"Seed {int(r['Seed'])}: "
        f"{r['Delta_Macro_F1']:+.6f} "
        f"({100*r['Delta_Macro_F1']:+.3f} percentage points)"
    )


## 12. Mean ± SD and stability comparison

In [ ]:

def model_summary(frame, model_name):
    s = frame[metric_cols].agg(["mean", "std"]).T
    s["Model"] = model_name
    s["CV_percent"] = 100.0 * s["std"] / s["mean"].abs()
    return s.reset_index(names="Metric")

adaptive_summary = model_summary(
    proposed_df,
    "URLFusionNet-Adaptive",
)
fixedavg_summary = model_summary(
    fixed_df,
    "Fixed-average fusion",
)

summary_comparison = pd.concat(
    [adaptive_summary, fixedavg_summary],
    ignore_index=True,
)

display(summary_comparison)
summary_comparison.to_csv(
    RESULT_DIR / "adaptive_vs_fixed_mean_std.csv",
    index=False,
)

a_f1_mean = float(
    adaptive_summary.loc[
        adaptive_summary["Metric"] == "Macro_F1", "mean"
    ].iloc[0]
)
a_f1_std = float(
    adaptive_summary.loc[
        adaptive_summary["Metric"] == "Macro_F1", "std"
    ].iloc[0]
)
f_f1_mean = float(
    fixedavg_summary.loc[
        fixedavg_summary["Metric"] == "Macro_F1", "mean"
    ].iloc[0]
)
f_f1_std = float(
    fixedavg_summary.loc[
        fixedavg_summary["Metric"] == "Macro_F1", "std"
    ].iloc[0]
)

wins = int((paired["Delta_Macro_F1"] > 0).sum())
ties = int(np.isclose(paired["Delta_Macro_F1"], 0, atol=1e-8).sum())

print("\n=== ADAPTIVE VS FIXED-AVERAGE ===")
print(f"Adaptive Macro-F1 : {a_f1_mean:.6f} ± {a_f1_std:.6f}")
print(f"FixedAvg Macro-F1 : {f_f1_mean:.6f} ± {f_f1_std:.6f}")
print(f"Mean difference   : {a_f1_mean - f_f1_mean:+.6f}")
print(f"Difference (pp)   : {100*(a_f1_mean - f_f1_mean):+.3f}")
print(f"Adaptive seed wins: {wins}/{len(SEEDS)}")
print(f"Ties              : {ties}/{len(SEEDS)}")
print(f"Adaptive F1 std   : {a_f1_std:.6f}")
print(f"FixedAvg F1 std   : {f_f1_std:.6f}")

if (a_f1_mean > f_f1_mean) and (a_f1_std <= f_f1_std):
    decision = (
        "Adaptive fusion has higher mean Macro-F1 and equal/better "
        "multi-seed stability than fixed averaging."
    )
elif a_f1_mean > f_f1_mean:
    decision = (
        "Adaptive fusion has higher mean Macro-F1, but it is not more "
        "stable than fixed averaging across the three seeds."
    )
elif a_f1_std < f_f1_std:
    decision = (
        "Adaptive fusion does not improve mean Macro-F1, although it is "
        "more stable across seeds. Do not claim a predictive-performance "
        "advantage over fixed averaging."
    )
else:
    decision = (
        "Adaptive fusion does not improve mean Macro-F1 or stability over "
        "fixed averaging. Position the adaptive gate primarily as an "
        "interpretable sample-dependent fusion mechanism, not as a source "
        "of superior average predictive performance."
    )

print("\nINTERPRETATION:", decision)

decision_df = pd.DataFrame([{
    "Adaptive_Macro_F1_Mean": a_f1_mean,
    "Adaptive_Macro_F1_SD": a_f1_std,
    "FixedAvg_Macro_F1_Mean": f_f1_mean,
    "FixedAvg_Macro_F1_SD": f_f1_std,
    "Mean_Delta": a_f1_mean - f_f1_mean,
    "Mean_Delta_Percentage_Points": 100*(a_f1_mean - f_f1_mean),
    "Adaptive_Wins_Seeds": wins,
    "Total_Seeds": len(SEEDS),
    "Interpretation": decision,
}])

display(decision_df)
decision_df.to_csv(
    RESULT_DIR / "adaptive_vs_fixed_decision.csv",
    index=False,
)


## 13. Visual paired comparison

In [ ]:

plot_df = paired.sort_values("Seed")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    plot_df["Seed"].astype(str),
    plot_df["Macro_F1_Adaptive"],
    marker="o",
    label="Adaptive fusion",
)
ax.plot(
    plot_df["Seed"].astype(str),
    plot_df["Macro_F1_FixedAvg"],
    marker="o",
    label="Fixed-average fusion",
)
ax.set_xlabel("Random seed")
ax.set_ylabel("Macro-F1")
ax.set_title("Adaptive vs Fixed-Average Fusion Across Identical Seeds")
ax.legend()
fig.tight_layout()
fig.savefig(
    RESULT_DIR / "fig_adaptive_vs_fixed_multiseed_macro_f1.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 14. Context against the strongest external/deep baseline

In [ ]:

BEST_BASELINE_NAME = "CNNTransformer_TCURLStyle"
BEST_BASELINE_MACRO_F1 = 0.9590648301547304

context = pd.DataFrame([
    {
        "Model": "URLFusionNet-Adaptive",
        "Macro_F1_mean": a_f1_mean,
        "Macro_F1_std": a_f1_std,
        "Delta_vs_best_baseline": a_f1_mean - BEST_BASELINE_MACRO_F1,
    },
    {
        "Model": "Fixed-average fusion",
        "Macro_F1_mean": f_f1_mean,
        "Macro_F1_std": f_f1_std,
        "Delta_vs_best_baseline": f_f1_mean - BEST_BASELINE_MACRO_F1,
    },
    {
        "Model": BEST_BASELINE_NAME,
        "Macro_F1_mean": BEST_BASELINE_MACRO_F1,
        "Macro_F1_std": np.nan,
        "Delta_vs_best_baseline": 0.0,
    },
])

display(context)
context.to_csv(
    RESULT_DIR / "context_vs_best_baseline.csv",
    index=False,
)



## 15. Results-readiness check

No significance claim is made from only three seed-level observations. The paper should report **mean ± SD, paired seed-wise differences, and the observed win/stability pattern**. Statistical significance should not be asserted from \(n=3\) seeds alone.


In [ ]:

required = {
    "FixedAvg seeds completed": set(fixed_df["Seed"].astype(int)) == set(SEEDS),
    "No non-finite FixedAvg metrics":
        np.isfinite(fixed_df[metric_cols].to_numpy()).all(),
    "Paired comparison has three seeds": len(paired) == 3,
    "Mean/std comparison generated":
        (RESULT_DIR / "adaptive_vs_fixed_mean_std.csv").exists(),
    "Decision table generated":
        (RESULT_DIR / "adaptive_vs_fixed_decision.csv").exists(),
    "Paired figure generated":
        (RESULT_DIR / "fig_adaptive_vs_fixed_multiseed_macro_f1.png").exists(),
}

check_df = pd.DataFrame([
    {"Requirement": k, "Passed": bool(v)}
    for k, v in required.items()
])

display(check_df)
check_df.to_csv(
    RESULT_DIR / "readiness_check.csv",
    index=False,
)

print("\nOutputs for the Results section:")
for p in sorted(RESULT_DIR.glob("*")):
    print(" -", p)



# What to send back

After this notebook finishes, upload the executed `.ipynb` or the folder `urlfusionnet_fixedavg_multiseed`.

The most important files are:

- `fixedavg_multiseed_raw.csv`
- `fixedavg_multiseed_mean_std.csv`
- `adaptive_vs_fixed_paired_by_seed.csv`
- `adaptive_vs_fixed_mean_std.csv`
- `adaptive_vs_fixed_decision.csv`
- `fig_adaptive_vs_fixed_multiseed_macro_f1.png`

From those results, the manuscript can make one of three evidence-based statements:

1. **Performance + stability support** — adaptive fusion has higher mean Macro-F1 and equal/lower SD.
2. **Performance-only support** — adaptive fusion has higher mean Macro-F1 but larger SD.
3. **No predictive advantage** — fixed averaging matches/exceeds adaptive fusion; adaptive fusion must then be positioned for sample-dependent behavior/interpretability rather than superior mean accuracy.

This prevents overstating the architectural novelty.
